# Workshop 3 — ETL Streaming con Apache Kafka
## Step 3 & 4: Feature Engineering + Entrenamiento del Modelo

**Curso:** ETL (G01) — Data Engineering and Artificial Intelligence  
**Input:** `data/processed/happiness_unified.csv`  
**Output:** `models/model.pkl`

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Librerías cargadas correctamente ✓')

---
## 1. Carga del dataset unificado

In [ ]:
df = pd.read_csv('../data/processed/happiness_unified.csv')

print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')
display(df.head())

---
## 2. Feature Engineering

### 2.1 Selección de features

**Target:** `happiness_score`

**Features seleccionadas:**

| Feature | Justificación |
|---|---|
| `gdp` | Alta correlación con felicidad (>0.8 en EDA) |
| `family` | Soporte social — fuerte predictor |
| `health` | Expectativa de vida — correlación alta |
| `freedom` | Libertad de elección — correlación moderada-alta |
| `generosity` | Incluida por completitud del modelo |
| `corruption` | Percepción institucional — correlación moderada |

**Features descartadas:**
- `country`: categórica con alta cardinalidad, no aporta al modelo de regresión simple
- `year`: no es un predictor causal de felicidad

**Sin target leakage:** ninguna feature es derivada directamente del happiness_score.

In [ ]:
FEATURES = ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
TARGET = 'happiness_score'

# Verificar que todas las features existen
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f'⚠️ Features faltantes: {missing}')
else:
    print('Todas las features disponibles ✓')

# Dataset final para ML
df_ml = df[FEATURES + [TARGET]].dropna()
print(f'Registros para entrenamiento: {df_ml.shape[0]}')
display(df_ml.describe())

### 2.2 Visualización de relaciones

In [ ]:
# Correlación de cada feature con el target
correlations = df_ml.corr()[TARGET].drop(TARGET).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
correlations.plot(kind='bar', color='steelblue')
plt.title('Correlación de features con Happiness Score')
plt.ylabel('Correlación')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print('\nCorrelaciones:')
print(correlations)

---
## 3. División train/test (70/30)

In [ ]:
X = df_ml[FEATURES]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f'Train: {X_train.shape[0]} registros ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]} registros ({X_test.shape[0]/len(X)*100:.1f}%)')

---
## 4. Entrenamiento y evaluación de modelos

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2   = r2_score(y_test, y_pred)
    
    print(f'\n{'='*40}')
    print(f'Modelo: {name}')
    print(f'  MAE:  {mae:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  R²:   {r2:.4f}')
    
    return {'name': name, 'model': model, 'mae': mae, 'rmse': rmse, 'r2': r2, 'y_pred': y_pred}

models = [
    ('Linear Regression',      LinearRegression()),
    ('Decision Tree',          DecisionTreeRegressor(random_state=42)),
    ('Random Forest',          RandomForestRegressor(n_estimators=100, random_state=42)),
]

results = []
for name, model in models:
    res = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(res)

### 4.1 Comparación de modelos

In [ ]:
metrics_df = pd.DataFrame([{
    'Modelo': r['name'],
    'MAE': round(r['mae'], 4),
    'RMSE': round(r['rmse'], 4),
    'R²': round(r['r2'], 4)
} for r in results])

display(metrics_df)

# Gráfico comparativo
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    ax.bar(metrics_df['Modelo'], metrics_df[metric], color=['steelblue', 'coral', 'seagreen'])
    ax.set_title(metric)
    ax.set_xticklabels(metrics_df['Modelo'], rotation=15, ha='right')

plt.suptitle('Comparación de modelos', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Selección y serialización del modelo final

**Criterio de selección:** mejor R² con menor RMSE.  
El taller no requiere optimización — se elige el modelo con mejores métricas base.

In [ ]:
# Seleccionar el mejor modelo por R²
best = max(results, key=lambda x: x['r2'])
print(f'Modelo seleccionado: {best["name"]}')
print(f'  R²:   {best["r2"]:.4f}')
print(f'  MAE:  {best["mae"]:.4f}')
print(f'  RMSE: {best["rmse"]:.4f}')

In [ ]:
# Guardar el modelo y la lista de features (necesaria para el consumer)
os.makedirs('../models', exist_ok=True)

model_artifact = {
    'model': best['model'],
    'features': FEATURES
}

with open('../models/model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print('Modelo guardado en ../models/model.pkl ✓')
print(f'Features del modelo: {FEATURES}')

---
## 6. Visualización: Predicted vs Actual

In [ ]:
y_pred_best = best['y_pred']

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Happiness Score')
plt.ylabel('Predicted Happiness Score')
plt.title(f'Predicted vs Actual — {best["name"]}')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución del error de predicción
errors = y_test.values - y_pred_best

plt.figure(figsize=(8, 4))
sns.histplot(errors, bins=30, kde=True, color='coral')
plt.axvline(0, color='black', linestyle='--')
plt.title('Distribución del error de predicción')
plt.xlabel('Error (actual - predicted)')
plt.tight_layout()
plt.show()

print(f'Error medio: {errors.mean():.4f}')
print(f'Desv. estándar del error: {errors.std():.4f}')

---
## 7. Verificación del modelo guardado

In [ ]:
# Cargar y probar el modelo guardado
with open('../models/model.pkl', 'rb') as f:
    artifact = pickle.load(f)

loaded_model = artifact['model']
loaded_features = artifact['features']

# Prueba con un evento de ejemplo (mismo formato que el Kafka producer)
test_event = {
    'country': 'Colombia',
    'year': 2019,
    'gdp': 1.2,
    'family': 0.8,
    'health': 0.9,
    'freedom': 0.6,
    'generosity': 0.3,
    'corruption': 0.1,
    'actual_happiness_score': 6.2
}

input_df = pd.DataFrame([[test_event[f] for f in loaded_features]], columns=loaded_features)
prediction = loaded_model.predict(input_df)[0]

print('Modelo cargado correctamente ✓')
print(f'Features esperadas: {loaded_features}')
print(f'\nEvento de prueba: Colombia 2019')
print(f'  Actual:    {test_event["actual_happiness_score"]}')
print(f'  Predicho:  {prediction:.4f}')
print(f'  Error:     {abs(test_event["actual_happiness_score"] - prediction):.4f}')